In [2]:
import pandas as pd
import duckdb

In [3]:
df = pd.read_excel(r'C:\Users\jdspr\OneDrive\Documents\Documents\Resume\SQL Practice\iowa_2023_2024.xlsx')

In [4]:
#add columns for year, month, day
df['year']=df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

In [5]:
#reorder columns 
cols=['invoice_and_item_number','date', 'year', 'month', 'day']
df=df[cols+[c for c in df.columns if c not in cols]]

In [6]:
#new dataframe with year, month, day columns and reorganized
#df

In [7]:
#initiate duckdb and recreate df as a table

In [8]:
con = duckdb.connect()
con.register("df_view", df)
con.execute("""
    CREATE 
        TABLE raw_sales AS SELECT * FROM df_view
    ;
""")

In [9]:
con.execute(
    """SELECT COUNT(*) 
    FROM raw_sales;"""
    ).df()


,count_star()
0,30352


In [10]:
#Count rows, min/max dates
con.execute(
    """SELECT 
        COUNT(*) AS rows,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM raw_sales;"""
    ).df()


,rows,min_date,max_date
0,30352,2023-01-03,2024-12-31


In [11]:
#for reference
df.columns

Index(['invoice_and_item_number', 'date', 'year', 'month', 'day',
       'store_number', 'store_name', 'address', 'city', 'zip_code',
       'store_location', 'county_number', 'county', 'category',
       'category_name', 'vendor_number', 'vendor_name', 'item_number',
       'item_description', 'pack', 'bottle_volume_ml', 'state_bottle_cost',
       'state_bottle_retail', 'bottles_sold', 'sale_dollars',
       'volume_sold_liters', 'volume_sold_gallons'],
      dtype='object')

In [12]:
#Top 10 stores by total volume
con.execute("""
    SELECT 
        store_number,store_name, SUM(volume_sold_gallons) AS total_volume_gal
    FROM raw_sales
    GROUP BY store_number,store_name
    ORDER BY total_volume_gal DESC
    LIMIT 10;
""").df()

,store_number,store_name,total_volume_gal
0,2633,HY-VEE #3 / BDI / DES MOINES,8031.19
1,4829,CENTRAL CITY 2,6432.02
2,3385,SAM'S CLUB 8162 / CEDAR RAPIDS,2980.13
3,2512,HY-VEE WINE AND SPIRITS #1 (1281) / IOWA CITY,2404.62
4,4677,COSTCO WHOLESALE #1111 / CORALVILLE,2381.95
5,3773,BENZ DISTRIBUTING,2344.15
6,5916,ANOTHER ROUND / DEWITT,2270.46
7,3494,SAM'S CLUB 6514 / WATERLOO,2159.90
8,5666,COSTCO WHOLESALE #1325 / DAVENPORT,1879.33
9,5144,SAM'S CLUB 6979 / ANKENY,1804.23


In [13]:
#Monthly total volume

con.execute("""
    SELECT 
       month, SUM(volume_sold_gallons) AS total_volume_gal
    FROM raw_sales
    GROUP BY month
    ;
""").df()

,month,total_volume_gal
0,1,5202.85
1,2,8173.25
2,3,10135.63
3,4,8332.27
4,5,8589.01
5,6,6925.95
6,7,8404.35
7,8,8110.01
8,9,9151.96
9,10,9401.44


In [14]:
#Monthly total volume for 2024

con.execute("""
    SELECT 
       month, SUM(volume_sold_gallons) AS total_volume_gal
    FROM raw_sales
    WHERE year = 2024
    GROUP BY month
    ;
""").df()

,month,total_volume_gal
0,1,2683.26
1,2,4808.45
2,3,4018.64
3,4,5154.05
4,5,3793.81
5,6,2627.85
6,7,4459.85
7,8,3408.73
8,9,4124.68
9,10,4033.68


In [15]:
con.execute("""
    SELECT 
        MAX(volume_sold_gallons) AS max_volume,
        MIN(volume_sold_gallons) AS min_volume
    FROM raw_sales 
    ;
""").df()

,max_volume,min_volume
0,2092.24,-8.32


In [16]:
con.execute("""
    SELECT 
        MAX(bottle_volume_ml) AS max_volume,
        MIN(bottle_volume_ml) AS min_volume
    FROM raw_sales
    ;
""").df()

,max_volume,min_volume
0,5250,20


In [17]:
#it's in bottles_sold
con.execute("""
    SELECT 
        MAX(bottles_sold) AS max_volume,
        MIN(bottles_sold) AS min_volume
    FROM raw_sales
    ;
""").df()

,max_volume,min_volume
0,7920,-48


In [18]:
#Create a Clean Base View, no negatives
con.execute("""
    CREATE OR REPLACE VIEW v_sales_clean AS
        SELECT
            CAST(date AS DATE) AS sale_date,
            DATE_TRUNC('month', CAST(date AS DATE)) AS sale_month,
            store_number,
            store_name,
            county,
            category_name,
            item_description,
            volume_sold_liters AS units,
            sale_dollars AS dollars,
        FROM raw_sales
        WHERE volume_sold_liters > 0;
""").df()

,Count


In [19]:
#DATE_TRUNC('month', CAST(date AS DATE)) AS sale_month locks the month in by setting day to 1

In [20]:
con.execute("""
    SELECT COUNT(*) 
    FROM v_sales_clean;
    """).df()

,count_star()
0,30209


In [21]:
con.execute("""
    SELECT *
    FROM v_sales_clean;
    """).df()

,sale_date,sale_month,store_number,store_name,county,category_name,item_description,units,dollars
0,2024-06-14,2024-06-01,2695,HY-VEE GAS #4 / DES MOINES,POLK,WHISKEY LIQUEUR,FIREBALL CINNAMON WHISKEY,14.4,216.00
1,2024-07-21,2024-07-01,5916,ANOTHER ROUND / DEWITT,CLINTON,IMPORTED SCHNAPPS,DR MCGILLICUDDYS MENTHOLMINT,52.5,675.00
2,2023-11-26,2023-11-01,2566,HY-VEE FOOD STORE (1353) / KNOXVILLE,MARION,AMERICAN VODKAS,SMIRNOFF 80PRF PET,472.5,5975.10
3,2024-09-19,2024-09-01,4829,CENTRAL CITY 2,POLK,AMERICAN VODKAS,TITOS HANDMADE VODKA,60.0,1185.60
4,2024-08-01,2024-08-01,4091,FAREWAY STORES #705 / CLEAR LAKE,CERRO GORDO,AMERICAN FLAVORED VODKA,UV BLUE RASPBERRY,31.5,229.50
...,...,...,...,...,...,...,...,...,...
30204,2023-11-03,2023-11-01,6108,DOUBLE D LIQUOR STORE / WAUKON,ALLAMAKEE,AMERICAN CORDIALS & LIQUEURS,EVAN WILLIAMS HONEY,1.5,27.00
30205,2024-05-06,2024-05-01,6236,WESTSIDE SPIRITS / CEDAR RAPIDS,LINN,AMERICAN DRY GINS,CLEARHEART GIN,1.5,36.90
30206,2023-04-14,2023-04-01,5409,CASEY'S GENERAL STORE # 2179/ WAUKEE,DALLAS,IMPORTED VODKAS,ABSOLUT SWEDISH VODKA 80PRF,1.5,29.98
30207,2024-08-30,2024-08-01,2649,HY-VEE #3 / DUBUQUE,DUBUQUE,IMPORTED CORDIALS & LIQUEURS,DISARONNO AMARETTO,1.5,47.46


In [22]:
#returns entire view :)

In [23]:
con.execute("""
   CREATE OR REPLACE VIEW v_monthly_scorecard AS
    SELECT
      county AS account,
      sale_month,
      SUM(units) AS total_units,
      SUM(dollars) AS total_dollars,
      COUNT(DISTINCT store_number) AS active_locations,
    FROM v_sales_clean,
    GROUP BY 1,2;
    """).df()

,Count


In [24]:
#GROUP BY 1,2 is equivalent to GROUP BY county, sale_month :)


In [25]:
#return entire view
con.execute("""
    SELECT *
    FROM v_monthly_scorecard;
    """).df()

,account,sale_month,total_units,total_dollars,active_locations
0,O'BRIEN,2023-02-01,37.95,580.54,4
1,JOHNSON,2023-08-01,413.39,14739.94,34
2,CLINTON,2023-01-01,495.32,9930.97,11
3,SCOTT,2023-11-01,557.95,13403.91,32
4,DALLAS,2024-07-01,524.47,15523.96,17
...,...,...,...,...,...
2164,OSCEOLA,2024-09-01,1.75,37.28,1
2165,IDA,2024-06-01,0.75,41.24,1
2166,OSCEOLA,2024-08-01,0.75,67.64,1
2167,AUDUBON,2023-08-01,1.12,25.20,1


In [26]:
con.execute("""
    SELECT
          *,
          total_units - LAG(total_units) OVER (PARTITION BY account ORDER BY sale_month) AS mom_units,
          (total_units - LAG(total_units) OVER (PARTITION BY account ORDER BY sale_month))
            / NULLIF(LAG(total_units) OVER (PARTITION BY account ORDER BY sale_month), 0) AS mom_pct,
          AVG(total_units) OVER (
            PARTITION BY account 
            ORDER BY sale_month 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
          ) AS rolling_3mo_units
        FROM v_monthly_scorecard;
""").df()

,account,sale_month,total_units,total_dollars,active_locations,mom_units,mom_pct,rolling_3mo_units
0,CLAY,2023-01-01,8.75,467.61,2,NaN,NaN,8.750000
1,CLAY,2023-02-01,95.15,705.15,2,86.40,9.874286,51.950000
2,CLAY,2023-03-01,53.74,711.98,1,-41.41,-0.435208,52.546667
3,CLAY,2023-04-01,43.85,575.59,3,-9.89,-0.184034,64.246667
4,CLAY,2023-05-01,8.56,417.53,3,-35.29,-0.804789,35.383333
...,...,...,...,...,...,...,...,...
2164,RINGGOLD,2023-02-01,0.10,73.80,1,NaN,NaN,0.100000
2165,RINGGOLD,2023-05-01,3.70,494.26,2,3.60,36.000000,1.900000
2166,RINGGOLD,2024-08-01,0.75,44.25,1,-2.95,-0.797297,1.516667
2167,RINGGOLD,2024-09-01,36.10,699.00,1,35.35,47.133333,13.516667


In [27]:
#LAG() moves a previous row value to the current, must be conditioned using OVER to calculate month-over-month numbers

In [28]:
con.execute("""
    WITH monthly AS (
      SELECT
        category_name,
        sale_month,
        SUM(units) AS units
      FROM v_sales_clean
      GROUP BY 1,2
    ),
    deltas AS (
      SELECT
        category_name,
        sale_month,
        units - LAG(units) OVER (PARTITION BY category_name ORDER BY sale_month) AS delta_units
      FROM monthly
    )
    SELECT *
    FROM deltas
    ORDER BY delta_units DESC
LIMIT 10;
""").df()

,category_name,sale_month,delta_units
0,IMPORTED DISTILLED SPIRITS SPECIALTY,2023-03-01,8116.00
1,SPICED RUM,2023-11-01,4922.46
2,AMERICAN VODKAS,2023-05-01,4898.93
3,AMERICAN VODKAS,2024-11-01,4232.41
4,SPICED RUM,2023-03-01,3962.61
5,CANADIAN WHISKIES,2024-07-01,3886.71
6,AMERICAN VODKAS,2024-02-01,3875.66
7,AMERICAN VODKAS,2023-10-01,3323.95
8,AMERICAN VODKAS,2023-12-01,3313.32
9,SPECIAL ORDER ITEMS,2024-04-01,3255.80


In [30]:
#Which category-month combinations had the largest month‑over‑month increase in units?

In [37]:
con.execute("""
   WITH current AS (
  SELECT DISTINCT county AS account, sale_month, store_number
  FROM v_sales_clean
),
prev AS (
  SELECT DISTINCT account, sale_month + INTERVAL '1 month' AS sale_month, store_number
  FROM current
)
SELECT
  c.account,
  c.sale_month,
  COUNT(c.store_number) FILTER (WHERE p.store_number IS NULL) AS gained,
  COUNT(p.store_number) FILTER (WHERE c.store_number IS NULL) AS lost
FROM current c
FULL JOIN prev p
  ON c.account = p.account
 AND c.sale_month = p.sale_month
 AND c.store_number = p.store_number
GROUP BY 1,2;

""").df()

,account,sale_month,gained,lost
0,POLK,2024-09-01,44,0
1,WASHINGTON,2024-08-01,2,0
2,STORY,2023-05-01,8,0
3,DALLAS,2023-10-01,7,0
4,MARSHALL,2024-10-01,4,0
...,...,...,...,...
2165,WINNEBAGO,2023-08-01,1,0
2166,CALHOUN,2024-07-01,1,0
2167,WINNEBAGO,2023-03-01,1,0
2168,IDA,2023-02-01,1,0


In [38]:
list(df.columns)

['invoice_and_item_number',
 'date',
 'year',
 'month',
 'day',
 'store_number',
 'store_name',
 'address',
 'city',
 'zip_code',
 'store_location',
 'county_number',
 'county',
 'category',
 'category_name',
 'vendor_number',
 'vendor_name',
 'item_number',
 'item_description',
 'pack',
 'bottle_volume_ml',
 'state_bottle_cost',
 'state_bottle_retail',
 'bottles_sold',
 'sale_dollars',
 'volume_sold_liters',
 'volume_sold_gallons']

In [39]:
con.execute("""
CREATE OR REPLACE VIEW v_sales_clean AS
SELECT
  -- identifiers
  invoice_and_item_number AS invoice_item_id,

  -- dates
  CAST(date AS DATE) AS sale_date,
  DATE_TRUNC('month', CAST(date AS DATE)) AS sale_month,
  CAST(year AS INTEGER) AS sale_year,
  CAST(month AS INTEGER) AS sale_month_num,
  CAST(day AS INTEGER) AS sale_day,

  -- store / location ("distribution" proxy)
  CAST(store_number AS INTEGER) AS store_id,
  store_name,
  address,
  city,
  zip_code,
  store_location,
  CAST(county_number AS INTEGER) AS county_id,
  county,

  -- product
  CAST(category AS BIGINT) AS category_id,
  category_name,
  CAST(vendor_number AS INTEGER) AS vendor_id,
  vendor_name,
  CAST(item_number AS BIGINT) AS item_id,
  item_description,
  CAST(pack AS INTEGER) AS pack,
  CAST(bottle_volume_ml AS INTEGER) AS bottle_volume_ml,
  CAST(state_bottle_cost AS DOUBLE) AS state_bottle_cost,
  CAST(state_bottle_retail AS DOUBLE) AS state_bottle_retail,

  -- measures
  CAST(bottles_sold AS DOUBLE) AS bottles_sold,
  CAST(sale_dollars AS DOUBLE) AS sale_dollars,
  CAST(volume_sold_liters AS DOUBLE) AS volume_liters,
  CAST(volume_sold_gallons AS DOUBLE) AS volume_gallons

FROM raw_sales
WHERE date IS NOT NULL
  AND CAST(volume_sold_liters AS DOUBLE) > 0
  AND CAST(sale_dollars AS DOUBLE) >= 0
""")
